# Fine-tuning de DistilBERT

Proyecto de fine-tuning de modelo de lenguaje listo para **Google Colab**.

**Pregunta:** ¿cómo fine-tunear DistilBERT para clasificación de sentimiento?

**Método:** Hugging Face Transformers

**Stack:** transformers, datasets, torch

Dataset de ejemplo (IMDB o simulado).

## 0. Setup

In [ ]:
!pip install -q transformers datasets torch

import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
print("OK")

## 1. Datos de ejemplo

In [ ]:
# Dataset pequeño de ejemplo
textos = [
    "Me encantó la película, excelente",
    "Pésima actuación, no la recomiendo",
    "Muy buena, la volvería a ver",
    "Aburrida y larga",
    "Increíble historia y actores",
    "No me gustó nada"
]
labels = [1, 0, 1, 0, 1, 0]

df = pd.DataFrame({"text": : textos, "label" : labels})
dataset = Dataset.from_pandas(df)
print(dataset)

## 2. Tokenización

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("Tokenizado")

## 3. Modelo

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
print("Modelo cargado")

## 4. Entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir="/tmp/results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=1,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()
print("Entrenamiento completado")

## 5. Predicción de ejemplo

In [ ]:
ejemplo = "Esta película es fantástica"
inputs = tokenizer(ejemplo, return_tensors="pt")
outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=-1).item()
print(f"Predicción: {'Positivo' if pred == 1 else 'Negativo'}")

## 6. Conclusiones

- Fine-tuning de DistilBERT es rápido incluso en CPU.
- Ideal para clasificación de texto en Colab.
- Se puede extender a datasets más grandes (IMDB, SST-2).

**Consejo:** usar GPU en Colab para entrenamientos más largos.